# Chapter 1.3: Arithmetic Intensity & Roofline Model

## Learning Objectives

By the end of this notebook, you will:

1. **Define** arithmetic intensity (FLOPs/byte) and calculate it for common operations
2. **Build** an interactive roofline model plot from scratch
3. **Calculate** arithmetic intensity for matrix multiplication of various sizes
4. **Analyze** why LLM prefill is compute-bound while decode is memory-bound
5. **Compare** roofline characteristics of A100 vs H100
6. **Place** real LLM operations (attention, FFN, LayerNorm) on the roofline

---

## Prerequisites
- Chapter 1.2 (GPU Memory Hierarchy)
- Understanding of FLOPs and memory bandwidth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part1/chapter_1.3_roofline_model.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part1/chapter_1.3_roofline_model.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
from typing import Dict, Tuple

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---

## 1. What is Arithmetic Intensity?

### 1.1 Definition

**Arithmetic Intensity (AI)** is the ratio of floating-point operations to bytes transferred from/to memory:

$$\text{Arithmetic Intensity} = \frac{\text{FLOPs}}{\text{Bytes transferred}} \quad [\text{FLOP/byte}]$$

This single number tells us whether an operation is:
- **Memory-bound** (low AI): Waiting for data from memory, compute units are idle
- **Compute-bound** (high AI): Compute units are fully utilized, memory is fast enough

### 1.2 Simple Examples

| Operation | FLOPs | Bytes | AI (FLOP/byte) | Bound |
|-----------|-------|-------|-----------------|-------|
| Vector add `c = a + b` | N | 3N * 4 = 12N | 1/12 ≈ 0.08 | Memory |
| Scalar multiply `b = k*a` | N | 2N * 4 = 8N | 1/8 = 0.125 | Memory |
| Dot product `s = a·b` | 2N | 2N * 4 = 8N | 1/4 = 0.25 | Memory |
| Matrix multiply (NxN) | 2N³ | 3N² * 4 = 12N² | N/6 | Compute (large N) |

**Key insight**: Matrix multiplication's AI grows with N! This is why GPUs love large matrix multiplies.

In [ ]:
# Calculate arithmetic intensity for different operations

def ai_vector_add(n, dtype_bytes=4):
    """c = a + b: N FLOPs, read 2N + write N elements"""
    flops = n
    bytes_transferred = 3 * n * dtype_bytes
    return flops / bytes_transferred

def ai_dot_product(n, dtype_bytes=4):
    """s = a . b: 2N FLOPs (N multiplies + N adds), read 2N elements"""
    flops = 2 * n
    bytes_transferred = 2 * n * dtype_bytes
    return flops / bytes_transferred

def ai_matrix_vector(m, n, dtype_bytes=4):
    """y = Ax: 2MN FLOPs, read MN + N, write M"""
    flops = 2 * m * n
    bytes_transferred = (m * n + n + m) * dtype_bytes
    return flops / bytes_transferred

def ai_matrix_multiply(m, n, k, dtype_bytes=4):
    """C = AB: 2MNK FLOPs, read MK + KN + write MN"""
    flops = 2 * m * n * k
    bytes_transferred = (m * k + k * n + m * n) * dtype_bytes
    return flops / bytes_transferred

def ai_softmax(n, dtype_bytes=4):
    """softmax: ~5N FLOPs (exp, sum, div), read N write N"""
    flops = 5 * n
    bytes_transferred = 2 * n * dtype_bytes
    return flops / bytes_transferred


print("Arithmetic Intensity for Common Operations (FP32)")
print("=" * 60)
print(f"  Vector add (N=1M):      {ai_vector_add(1e6):.4f} FLOP/byte")
print(f"  Dot product (N=1M):     {ai_dot_product(1e6):.4f} FLOP/byte")
print(f"  MatVec (4096x4096):     {ai_matrix_vector(4096, 4096):.4f} FLOP/byte")
print(f"  Softmax (N=4096):       {ai_softmax(4096):.4f} FLOP/byte")
print()
print("  MatMul sizes:")
for n in [32, 128, 512, 1024, 4096, 8192]:
    ai = ai_matrix_multiply(n, n, n)
    print(f"    ({n}x{n}) x ({n}x{n}):   {ai:.1f} FLOP/byte")

---

## 2. The Roofline Model

### 2.1 Core Concept

The **Roofline Model** provides an upper bound on performance based on two hardware limits:

1. **Peak compute throughput** ($\pi$): Maximum FLOPS the GPU can perform
2. **Peak memory bandwidth** ($\beta$): Maximum bytes/second the GPU can transfer

For a kernel with arithmetic intensity $I$ (FLOP/byte), the achievable performance is:

$$P = \min(\pi, \beta \times I)$$

Where:
- $P$ = achievable FLOPS
- $\pi$ = peak compute (FLOPS)
- $\beta$ = peak bandwidth (bytes/s)
- $I$ = arithmetic intensity (FLOP/byte)

The **ridge point** is where these two lines meet:
$$I_{\text{ridge}} = \frac{\pi}{\beta}$$

- If $I < I_{\text{ridge}}$: **memory-bound** (performance = $\beta \times I$)
- If $I > I_{\text{ridge}}$: **compute-bound** (performance = $\pi$)

In [ ]:
## Figure: Roofline Model with LLM Prefill and Decode Regions
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(14, 8))

# A100 specs
peak_tflops = 312  # FP16 Tensor Core
peak_bw_tbs = 2.0

# Roofline
ai_range = np.logspace(-2, 4, 1000)
perf = np.minimum(peak_tflops * 1e12, peak_bw_tbs * 1e12 * ai_range) / 1e12
ax.loglog(ai_range, perf, 'b-', linewidth=3, alpha=0.8, label='A100 Roofline (FP16 TC)')

# Ridge point
ridge = peak_tflops / (peak_bw_tbs * 1000)
ax.axvline(x=ridge*1000, color='gray', linestyle=':', alpha=0.5)
ax.plot(ridge*1000, peak_tflops, 'ko', markersize=12, zorder=5)
ax.annotate(f'Ridge Point\n{ridge*1000:.0f} FLOP/B', xy=(ridge*1000, peak_tflops),
            xytext=(ridge*2000, peak_tflops*0.3), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Shade regions
ax.fill_between(ai_range[ai_range < ridge*1000], 0.01, 
                np.minimum(peak_tflops, peak_bw_tbs*1000*ai_range[ai_range < ridge*1000]),
                alpha=0.08, color='#E74C3C')
ax.fill_between(ai_range[ai_range >= ridge*1000], 0.01, peak_tflops,
                alpha=0.08, color='#27AE60')

ax.text(1, 0.15, 'MEMORY-BOUND\nREGION', fontsize=14, fontweight='bold', color='#E74C3C', alpha=0.7)
ax.text(1500, 0.15, 'COMPUTE-BOUND\nREGION', fontsize=14, fontweight='bold', color='#27AE60', alpha=0.7)

# LLM operations
ops = {
    'Decode\n(batch=1)': (0.5, 1.0, '#E74C3C', 's', 200),
    'Decode\n(batch=32)': (16, 32, '#F39C12', 's', 200),
    'Decode\n(batch=128)': (64, 128, '#F39C12', 's', 200),
    'Prefill\n(seq=512)': (170, 280, '#27AE60', '^', 250),
    'Prefill\n(seq=2048)': (680, 310, '#27AE60', '^', 250),
    'LayerNorm': (0.3, 0.6, '#8E44AD', 'D', 150),
    'Softmax': (0.6, 1.2, '#8E44AD', 'D', 150),
}

for name, (ai, perf_val, color, marker, sz) in ops.items():
    ax.plot(ai, perf_val, marker, color=color, markersize=14, markeredgecolor='black',
            markeredgewidth=1.5, zorder=5)
    offset_x, offset_y = 1.5, 1.3
    ax.annotate(name, xy=(ai, perf_val), xytext=(ai*offset_x, perf_val*offset_y),
                fontsize=9, fontweight='bold', color=color,
                arrowprops=dict(arrowstyle='->', color=color, alpha=0.7))

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=13)
ax.set_ylabel('Performance (TFLOPS)', fontsize=13)
ax.set_title('Roofline Model: Where LLM Operations Fall (A100 GPU)', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0.05, 5000)
ax.set_ylim(0.05, 500)

plt.tight_layout()
plt.show()
print("Decode (batch=1) is ~300x below the ridge point -- severely memory-bound!")
print("Prefill is near or above the ridge point -- compute-bound.")
print("Batching moves decode operations rightward along the memory-bound slope.")

In [ ]:
def plot_roofline(
    peak_compute_tflops: float,
    peak_bandwidth_tbs: float,
    gpu_name: str,
    operations: dict = None,
    ax=None,
    color='blue'
):
    """
    Plot the roofline model for a given GPU.
    
    operations: dict of {name: (arithmetic_intensity, achieved_tflops)} or
                {name: arithmetic_intensity} to just show the point on the roof
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 7))
    
    peak_flops = peak_compute_tflops * 1e12
    peak_bw = peak_bandwidth_tbs * 1e12
    
    # Ridge point
    ridge_point = peak_flops / peak_bw  # FLOP/byte
    
    # X-axis: arithmetic intensity (log scale)
    ai_range = np.logspace(-2, 4, 1000)
    
    # Roofline
    perf = np.minimum(peak_flops, peak_bw * ai_range)
    perf_tflops = perf / 1e12
    
    # Plot
    ax.loglog(ai_range, perf_tflops, color=color, linewidth=3, label=f'{gpu_name} Roofline')
    
    # Mark the ridge point
    ax.axvline(x=ridge_point, color=color, linestyle=':', alpha=0.5)
    ax.plot(ridge_point, peak_compute_tflops, 'o', color=color, markersize=10)
    ax.annotate(f'Ridge: {ridge_point:.0f} FLOP/B',
                xy=(ridge_point, peak_compute_tflops),
                xytext=(ridge_point * 2, peak_compute_tflops * 0.5),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, alpha=0.7))
    
    # Label regions
    ax.text(ridge_point * 0.05, peak_compute_tflops * 0.3, 'Memory\nBound',
            fontsize=12, ha='center', color='red', alpha=0.7, fontweight='bold')
    ax.text(ridge_point * 20, peak_compute_tflops * 0.3, 'Compute\nBound',
            fontsize=12, ha='center', color='green', alpha=0.7, fontweight='bold')
    
    # Plot operations if provided
    if operations:
        markers = ['s', '^', 'D', 'v', 'p', '*', 'h', '8']
        colors_ops = plt.cm.tab10(np.linspace(0, 1, len(operations)))
        
        for i, (name, data) in enumerate(operations.items()):
            if isinstance(data, tuple):
                ai, achieved = data
            else:
                ai = data
                achieved = min(peak_compute_tflops, peak_bandwidth_tbs * ai) * 0.7  # estimated
            
            marker = markers[i % len(markers)]
            ax.plot(ai, achieved, marker, color=colors_ops[i], markersize=12,
                    label=f'{name} (AI={ai:.1f})', markeredgecolor='black', markeredgewidth=1)
    
    ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=13)
    ax.set_ylabel('Performance (TFLOPS)', fontsize=13)
    ax.set_title(f'Roofline Model — {gpu_name}', fontsize=15)
    ax.legend(fontsize=9, loc='lower right')
    ax.grid(True, alpha=0.3, which='both')
    ax.set_xlim(0.01, 10000)
    ax.set_ylim(0.01, peak_compute_tflops * 2)
    
    return ax


# A100 Roofline
fig, ax = plt.subplots(figsize=(14, 8))
plot_roofline(
    peak_compute_tflops=312,  # FP16 Tensor Core
    peak_bandwidth_tbs=2.0,
    gpu_name='A100 (FP16 Tensor Core)',
    operations={
        'Vector Add': 0.083,
        'LayerNorm': 0.3,
        'MatVec 4096': ai_matrix_vector(4096, 4096, 2),
        'Softmax': ai_softmax(4096, 2),
        'MatMul 512': ai_matrix_multiply(512, 512, 512, 2),
        'MatMul 4096': ai_matrix_multiply(4096, 4096, 4096, 2),
    },
    ax=ax
)
plt.tight_layout()
plt.show()

### 2.2 Comparing A100 vs H100 Rooflines

The H100 has both higher compute and higher bandwidth. Let's see how the rooflines compare.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

# A100
plot_roofline(312, 2.0, 'A100 (FP16 TC)', ax=ax, color='blue')

# H100  
peak_bw_h100 = 3.35
peak_compute_h100 = 989
ai_range = np.logspace(-2, 4, 1000)
perf_h100 = np.minimum(peak_compute_h100 * 1e12, peak_bw_h100 * 1e12 * ai_range) / 1e12
ax.loglog(ai_range, perf_h100, color='red', linewidth=3, label='H100 (FP16 TC) Roofline')
ridge_h100 = peak_compute_h100 / peak_bw_h100 / 1000
ax.plot(ridge_h100 * 1000, peak_compute_h100, 'o', color='red', markersize=10)

# H200
peak_bw_h200 = 4.8
peak_compute_h200 = 989
perf_h200 = np.minimum(peak_compute_h200 * 1e12, peak_bw_h200 * 1e12 * ai_range) / 1e12
ax.loglog(ai_range, perf_h200, color='green', linewidth=3, label='H200 (FP16 TC) Roofline')

# Mark LLM operations
llm_ops = {
    'Decode\n(batch=1)': (0.5, 1.0),
    'Decode\n(batch=32)': (16, 50),
    'Prefill\n(seq=2K)': (341, 280),
}

for name, (ai, perf) in llm_ops.items():
    ax.plot(ai, perf, '*', markersize=18, markeredgecolor='black', markeredgewidth=1.5)
    ax.annotate(name, xy=(ai, perf), xytext=(ai * 1.5, perf * 1.5),
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black'))

ax.set_ylim(0.1, 2000)
ax.legend(fontsize=10, loc='lower right')
ax.set_title('Roofline Model Comparison: A100 vs H100 vs H200', fontsize=15)
plt.tight_layout()
plt.show()

print("Key observations:")
print(f"  A100 ridge point: {312/2.0:.0f} FLOP/byte")
print(f"  H100 ridge point: {989/3.35:.0f} FLOP/byte")
print(f"  H200 ridge point: {989/4.8:.0f} FLOP/byte")
print()
print("  H200 has a LOWER ridge point than H100 (more bandwidth per FLOP).")
print("  This makes H200 relatively better for memory-bound workloads like LLM decode!")

---

## 3. Arithmetic Intensity of Matrix Multiplication

### 3.1 General Formula

For $C_{m \times n} = A_{m \times k} \times B_{k \times n}$:

$$\text{FLOPs} = 2mnk$$
$$\text{Bytes} = (mk + kn + mn) \times \text{sizeof(dtype)}$$
$$\text{AI} = \frac{2mnk}{(mk + kn + mn) \times \text{sizeof}}$$

For square matrices ($m = n = k = N$):
$$\text{AI} = \frac{2N^3}{3N^2 \times \text{sizeof}} = \frac{2N}{3 \times \text{sizeof}}$$

So for FP16 (2 bytes): $\text{AI} = N/3$

In [ ]:
# How AI varies with matrix dimensions

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Square matrix: AI vs N
N_values = np.arange(16, 8193, 16)
ai_fp16 = [ai_matrix_multiply(n, n, n, dtype_bytes=2) for n in N_values]
ai_fp32 = [ai_matrix_multiply(n, n, n, dtype_bytes=4) for n in N_values]

axes[0].plot(N_values, ai_fp16, 'b-', linewidth=2, label='FP16')
axes[0].plot(N_values, ai_fp32, 'r-', linewidth=2, label='FP32')

# Ridge points
axes[0].axhline(y=312/2.0, color='blue', linestyle='--', alpha=0.5, label=f'A100 ridge (FP16): {312/2.0:.0f}')
axes[0].axhline(y=989/3.35, color='green', linestyle='--', alpha=0.5, label=f'H100 ridge (FP16): {989/3.35:.0f}')

axes[0].set_xlabel('Matrix Size N (NxN * NxN)', fontsize=12)
axes[0].set_ylabel('Arithmetic Intensity (FLOP/byte)', fontsize=12)
axes[0].set_title('AI of Square Matrix Multiply vs Size', fontsize=14)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Non-square: batch=1 decode (weight * 1 vector) vs prefill
# Decode: (4096, 4096) x (4096, 1) — matrix-vector
# Prefill: (4096, 4096) x (4096, seq_len)
seq_lengths = np.arange(1, 4097, 8)
d_model = 4096
ai_by_seq = [ai_matrix_multiply(d_model, int(s), d_model, dtype_bytes=2) for s in seq_lengths]

axes[1].plot(seq_lengths, ai_by_seq, 'b-', linewidth=2)
axes[1].axhline(y=312/2.0, color='red', linestyle='--', alpha=0.5, label=f'A100 ridge: {312/2.0:.0f}')
axes[1].axvline(x=1, color='green', linestyle=':', alpha=0.5)
axes[1].annotate('Decode\n(seq=1)', xy=(1, ai_by_seq[0]), xytext=(50, ai_by_seq[0]*2),
                 fontsize=10, arrowprops=dict(arrowstyle='->', color='red'),
                 color='red', fontweight='bold')
axes[1].annotate('Prefill\n(seq=2048)', xy=(2048, ai_by_seq[len(seq_lengths)//2]),
                 xytext=(2500, ai_by_seq[len(seq_lengths)//2]*0.5),
                 fontsize=10, arrowprops=dict(arrowstyle='->', color='green'),
                 color='green', fontweight='bold')

axes[1].set_xlabel('Sequence Length (batch dimension)', fontsize=12)
axes[1].set_ylabel('Arithmetic Intensity (FLOP/byte)', fontsize=12)
axes[1].set_title('AI of Linear Layer (4096x4096) vs Sequence Length', fontsize=14)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Decode (seq=1):    AI = {ai_matrix_multiply(4096, 1, 4096, 2):.1f} FLOP/byte  -> MEMORY-BOUND")
print(f"Prefill (seq=2048): AI = {ai_matrix_multiply(4096, 2048, 4096, 2):.1f} FLOP/byte -> COMPUTE-BOUND")

---

## 4. LLM Inference: Prefill vs Decode on the Roofline

### 4.1 Prefill Phase Analysis

During prefill, we process `seq_len` tokens simultaneously through each layer:

**Linear projections** (Q, K, V, O projections + FFN):
- Operation: $Y_{\text{seq} \times d} = X_{\text{seq} \times d} \times W_{d \times d}$
- FLOPs: $2 \times \text{seq} \times d^2$
- Bytes: $(\text{seq} \times d + d^2 + \text{seq} \times d) \times 2$ (FP16)
- AI: $\frac{2 \times \text{seq} \times d^2}{(2 \times \text{seq} \times d + d^2) \times 2}$

For large seq_len, AI $\approx d/2$ which is well above the ridge point.

### 4.2 Decode Phase Analysis

During decode, we process just 1 token:
- Same operation but seq=1: $y_{1 \times d} = x_{1 \times d} \times W_{d \times d}$  
- AI: $\frac{2d^2}{(2d + d^2) \times 2} \approx \frac{2d^2}{2d^2} = 1$ FLOP/byte

This is FAR below the ridge point!

In [ ]:
def analyze_llm_layer_ai(
    d_model: int = 4096,
    n_heads: int = 32,
    d_ff: int = 11008,  # Llama-2-7B FFN hidden dim
    seq_len: int = 1,
    batch_size: int = 1,
    dtype_bytes: int = 2  # FP16
) -> dict:
    """
    Calculate arithmetic intensity for each operation in a Transformer layer.
    """
    effective_batch = batch_size * seq_len  # total tokens processed
    d_head = d_model // n_heads
    results = {}
    
    # 1. Q, K, V projections (3 linear layers)
    for name in ['Q_proj', 'K_proj', 'V_proj']:
        flops = 2 * effective_batch * d_model * d_model
        # Read: input (batch * d_model) + weight (d_model * d_model)
        # Write: output (batch * d_model)
        bytes_rw = (effective_batch * d_model + d_model * d_model + effective_batch * d_model) * dtype_bytes
        results[name] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    # 2. Attention: QK^T
    flops = 2 * n_heads * effective_batch * effective_batch * d_head
    bytes_rw = (2 * n_heads * effective_batch * d_head + n_heads * effective_batch * effective_batch) * dtype_bytes
    results['QK^T'] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    # 3. Softmax
    flops = 5 * n_heads * effective_batch * effective_batch
    bytes_rw = 2 * n_heads * effective_batch * effective_batch * dtype_bytes
    results['Softmax'] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    # 4. Attention @ V
    flops = 2 * n_heads * effective_batch * effective_batch * d_head
    bytes_rw = (n_heads * effective_batch * effective_batch + n_heads * effective_batch * d_head + n_heads * effective_batch * d_head) * dtype_bytes
    results['Attn@V'] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    # 5. Output projection
    flops = 2 * effective_batch * d_model * d_model
    bytes_rw = (effective_batch * d_model + d_model * d_model + effective_batch * d_model) * dtype_bytes
    results['O_proj'] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    # 6. FFN (gate + up projections)
    flops = 2 * 2 * effective_batch * d_model * d_ff  # gate and up
    bytes_rw = (effective_batch * d_model + 2 * d_model * d_ff + 2 * effective_batch * d_ff) * dtype_bytes
    results['FFN_up'] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    # 7. FFN down projection
    flops = 2 * effective_batch * d_ff * d_model
    bytes_rw = (effective_batch * d_ff + d_ff * d_model + effective_batch * d_model) * dtype_bytes
    results['FFN_down'] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    # 8. Layer Norm (RMSNorm)
    flops = 5 * effective_batch * d_model  # approximate
    bytes_rw = 2 * effective_batch * d_model * dtype_bytes + d_model * dtype_bytes
    results['RMSNorm'] = {'flops': flops, 'bytes': bytes_rw, 'ai': flops / bytes_rw}
    
    return results


# Prefill: seq_len=2048
prefill_ai = analyze_llm_layer_ai(seq_len=2048, batch_size=1)
# Decode: seq_len=1, batch_size=1
decode_ai = analyze_llm_layer_ai(seq_len=1, batch_size=1)

print("Arithmetic Intensity per Operation (Llama-2-7B, FP16)")
print("=" * 70)
print(f"{'Operation':<15} {'Prefill (seq=2048)':<25} {'Decode (batch=1)':<25} {'Bound (decode)'}")
print("-" * 70)

ridge_a100 = 312 / 2.0  # = 156
for op in prefill_ai:
    p_ai = prefill_ai[op]['ai']
    d_ai = decode_ai[op]['ai']
    bound = 'MEMORY' if d_ai < ridge_a100 else 'COMPUTE'
    print(f"{op:<15} {p_ai:<25.1f} {d_ai:<25.2f} {bound}")

In [ ]:
# Plot LLM operations on the roofline for both prefill and decode

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Prefill roofline
prefill_ops = {name: data['ai'] for name, data in prefill_ai.items()}
plot_roofline(312, 2.0, 'A100 — Prefill (seq=2048)', operations=prefill_ops, ax=axes[0])
axes[0].set_title('Roofline: Prefill Phase (seq=2048)', fontsize=14)

# Decode roofline
decode_ops = {name: data['ai'] for name, data in decode_ai.items()}
plot_roofline(312, 2.0, 'A100 — Decode (batch=1)', operations=decode_ops, ax=axes[1])
axes[1].set_title('Roofline: Decode Phase (batch=1)', fontsize=14)

plt.tight_layout()
plt.show()

print("\nPrefill: Most operations are in the compute-bound region (right side).")
print("Decode:  ALL operations are in the memory-bound region (left side).")
print("\nThis is THE fundamental challenge of LLM serving!")

---

## 5. How Batching Moves Operations on the Roofline

Increasing the batch size during decode increases the arithmetic intensity of linear layers because we amortize reading the weight matrix across more tokens.

In [ ]:
# Show how batch size moves the linear layer on the roofline

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

fig, ax = plt.subplots(figsize=(14, 8))

# Plot A100 roofline
ai_range = np.logspace(-2, 4, 1000)
peak_flops = 312e12
peak_bw = 2.0e12
perf = np.minimum(peak_flops, peak_bw * ai_range) / 1e12
ax.loglog(ai_range, perf, 'b-', linewidth=3, label='A100 Roofline')

# Track linear layer AI as batch size increases
d_model = 4096
ais = []
perfs_theory = []

for bs in batch_sizes:
    result = analyze_llm_layer_ai(seq_len=1, batch_size=bs)
    ai = result['Q_proj']['ai']
    ais.append(ai)
    p = min(312, 2.0 * ai)  # theoretical peak (TFLOPS)
    perfs_theory.append(p)

# Plot the trajectory
ax.plot(ais, perfs_theory, 'ro-', markersize=10, linewidth=2, label='Linear Layer (Q proj)', 
        markeredgecolor='black', markeredgewidth=1)

for bs, ai, perf_t in zip(batch_sizes, ais, perfs_theory):
    ax.annotate(f'B={bs}', xy=(ai, perf_t), xytext=(ai*1.3, perf_t*1.2),
                fontsize=8, color='red')

ax.axvline(x=312/2.0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=13)
ax.set_ylabel('Performance (TFLOPS)', fontsize=13)
ax.set_title('Effect of Batch Size on Arithmetic Intensity (Decode)', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0.1, 5000)
ax.set_ylim(0.1, 500)

plt.tight_layout()
plt.show()

print("As batch size increases:")
print("  - AI increases (more FLOPs per byte of weights read)")
print("  - Performance climbs along the memory-bound slope")
print(f"  - At batch={batch_sizes[-1]}, we approach the compute-bound regime")
print(f"\nTo reach the A100 ridge point ({312/2.0:.0f} FLOP/byte):")
print(f"  Need batch >= ~{int(312/2.0 * 2 / (4096) * (4096*4096*2 + 4096*2) / (2*4096*4096))}") 

---

## 6. Measured vs Theoretical Performance

Let's measure actual performance for matrix multiplications of different sizes and overlay them on the roofline.

In [ ]:
def benchmark_matmul(M, N, K, dtype=torch.float16, device=device, n_iter=100, warmup=20):
    """Benchmark matrix multiplication and return TFLOPS and AI."""
    A = torch.randn(M, K, device=device, dtype=dtype)
    B = torch.randn(K, N, device=device, dtype=dtype)
    
    # Warmup
    for _ in range(warmup):
        C = torch.mm(A, B)
    if device.type == 'cuda': torch.cuda.synchronize()
    
    # Measure
    start = time.perf_counter()
    for _ in range(n_iter):
        C = torch.mm(A, B)
    if device.type == 'cuda': torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_iter
    
    flops = 2 * M * N * K
    dtype_bytes = 2 if dtype == torch.float16 else 4
    bytes_rw = (M * K + K * N + M * N) * dtype_bytes
    
    tflops = flops / elapsed / 1e12
    ai = flops / bytes_rw
    bw_gbs = bytes_rw / elapsed / 1e9
    
    del A, B, C
    if device.type == 'cuda': torch.cuda.empty_cache()
    
    return {'tflops': tflops, 'ai': ai, 'bw_gbs': bw_gbs, 'time_ms': elapsed * 1000}


# Benchmark different matrix sizes
configs = [
    # (M, N, K, description)
    (4096, 1, 4096, 'MatVec (decode b=1)'),
    (4096, 4, 4096, 'batch=4'),
    (4096, 16, 4096, 'batch=16'),
    (4096, 64, 4096, 'batch=64'),
    (4096, 256, 4096, 'batch=256'),
    (4096, 1024, 4096, 'batch=1024'),
    (4096, 4096, 4096, 'square 4K'),
]

print(f"Benchmarking on {device}...")
print(f"{'Config':<25} {'AI (FLOP/B)':<15} {'TFLOPS':<12} {'BW (GB/s)':<12} {'Time (ms)':<12}")
print("-" * 76)

measured_points = []
for M, N, K, desc in configs:
    result = benchmark_matmul(M, N, K)
    measured_points.append((desc, result['ai'], result['tflops']))
    print(f"{desc:<25} {result['ai']:<15.1f} {result['tflops']:<12.2f} {result['bw_gbs']:<12.1f} {result['time_ms']:<12.3f}")

In [ ]:
# Plot measured points on roofline

fig, ax = plt.subplots(figsize=(14, 8))

# Theoretical roofline
ai_range = np.logspace(-1, 4, 1000)

# Detect GPU for proper roofline
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    # Use conservative estimates for measured comparison
    peak_tflops = 312 if 'A100' in gpu_name else (989 if 'H100' in gpu_name else 100)
    peak_bw_tbs = 2.0 if 'A100' in gpu_name else (3.35 if 'H100' in gpu_name else 1.0)
else:
    gpu_name = 'CPU'
    peak_tflops = 1.0  # rough estimate
    peak_bw_tbs = 0.05

perf_roof = np.minimum(peak_tflops * 1e12, peak_bw_tbs * 1e12 * ai_range) / 1e12
ax.loglog(ai_range, perf_roof, 'b-', linewidth=3, alpha=0.7, label=f'{gpu_name} Roofline')

# Measured points
for desc, ai, tflops in measured_points:
    ax.plot(ai, tflops, 's', markersize=12, markeredgecolor='black', markeredgewidth=1.5)
    ax.annotate(desc, xy=(ai, tflops), xytext=(ai * 1.5, tflops * 0.7),
                fontsize=9, arrowprops=dict(arrowstyle='->', alpha=0.5))

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=13)
ax.set_ylabel('Performance (TFLOPS)', fontsize=13)
ax.set_title(f'Measured Performance on Roofline — {gpu_name}', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\nPoints below the roofline indicate less-than-peak utilization.")
print("The gap is due to: kernel launch overhead, non-ideal tiling, etc.")

---

## 7. Implications for LLM Serving Optimization

### 7.1 Optimization Strategies Based on Roofline Position

| Regime | Problem | Solutions |
|--------|---------|----------|
| Memory-bound (decode) | GPU idle, waiting for data | Quantization, batching, KV-cache compression |
| Compute-bound (prefill) | Memory fast enough | Faster kernels, FlashAttention, tensor parallelism |
| Balanced (large batch decode) | Near optimal | Fine-tune batch size |

### 7.2 Why Quantization Helps Decode

Quantization (FP16 → INT8 → INT4) reduces the number of bytes to read from HBM:
- Doubles/quadruples the effective arithmetic intensity
- Moves the operation closer to (or past) the ridge point
- Nearly doubles/quadruples decode throughput

In [ ]:
# Show how quantization shifts operations on the roofline

fig, ax = plt.subplots(figsize=(14, 8))

# A100 roofline
ai_range = np.logspace(-1, 4, 1000)
perf_roof = np.minimum(312e12, 2.0e12 * ai_range) / 1e12
ax.loglog(ai_range, perf_roof, 'b-', linewidth=3, alpha=0.7, label='A100 Roofline (FP16)')

# Linear layer (4096x4096) at batch=1 with different quantizations
d = 4096
quant_configs = [
    ('FP32 (4B)', 4, 'red'),
    ('FP16 (2B)', 2, 'orange'),
    ('INT8 (1B)', 1, 'green'),
    ('INT4 (0.5B)', 0.5, 'blue'),
]

for name, dtype_bytes, color in quant_configs:
    flops = 2 * d * d  # FLOPs don't change with quantization
    bytes_rw = (d + d * d + d) * dtype_bytes  # weights dominate
    ai = flops / bytes_rw
    perf = min(312, 2000 * ai)  # GB/s, convert to TFLOPS
    
    ax.plot(ai, perf, 'o', markersize=14, color=color, markeredgecolor='black',
            markeredgewidth=2, label=f'{name}: AI={ai:.1f}', zorder=5)

# Arrow showing the effect
ax.annotate('', xy=(ai_matrix_multiply(d, 1, d, 0.5), min(312, 2000*ai_matrix_multiply(d, 1, d, 0.5))),
            xytext=(ai_matrix_multiply(d, 1, d, 4), min(312, 2000*ai_matrix_multiply(d, 1, d, 4))),
            arrowprops=dict(arrowstyle='->', color='purple', linewidth=3, alpha=0.5))
ax.text(2, 10, 'Quantization\nshifts right!', fontsize=12, color='purple', fontweight='bold',
        ha='center')

ax.set_xlabel('Arithmetic Intensity (FLOP/byte)', fontsize=13)
ax.set_ylabel('Performance (TFLOPS)', fontsize=13)
ax.set_title('Effect of Quantization on Roofline Position (batch=1 decode)', fontsize=15)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0.05, 5000)
ax.set_ylim(0.1, 500)

plt.tight_layout()
plt.show()

print("Quantization increases AI by reducing bytes transferred:")
for name, dtype_bytes, _ in quant_configs:
    ai = ai_matrix_multiply(d, 1, d, dtype_bytes)
    tok_per_s = 2000 * ai / (2 * d * d) * 1e12  # rough estimate
    print(f"  {name:<12} AI = {ai:.1f}  → theoretical ~{min(312, 2000*ai):.1f} TFLOPS")

---

## Exercises

### Exercise 1: Build a Custom Roofline

Create a roofline plot for a custom GPU or accelerator with the following specs, and plot these LLM operations on it.

In [ ]:
# Exercise 1: Custom Roofline

# Given:
# - A custom AI chip with:
#   - 500 TFLOPS FP16 compute
#   - 1.5 TB/s HBM bandwidth
#
# 1. What is the ridge point?
# 2. Plot the roofline
# 3. For Llama-2-7B (d_model=4096):
#    a. Is prefill (seq=1024) compute-bound or memory-bound?
#    b. Is decode (batch=1) compute-bound or memory-bound?
#    c. What batch size makes decode compute-bound?

# YOUR CODE HERE
# ridge = 500 / 1.5  # = ?
# ...

### Exercise 2: Calculate Roofline for a Specific Model

For a Llama-3-70B model with GQA (8 KV heads, 64 Q heads, d_head=128, 80 layers):

1. Calculate the AI of the attention computation during decode with KV-cache length=2048
2. Is the attention memory-bound or compute-bound on H100?
3. How does the AI change with KV-cache length?

In [ ]:
# Exercise 2: Attention AI analysis for Llama-3-70B

def attention_ai_decode(
    n_q_heads: int,
    n_kv_heads: int,
    d_head: int,
    kv_cache_len: int,
    dtype_bytes: int = 2
):
    """
    Exercise: Calculate AI for attention during decode.
    
    During decode:
    - Q: (n_q_heads, 1, d_head)
    - K cache: (n_kv_heads, kv_cache_len, d_head)
    - V cache: (n_kv_heads, kv_cache_len, d_head)
    
    QK^T: each Q head attends to its corresponding KV head
    FLOPs for QK^T: n_q_heads * 2 * 1 * kv_cache_len * d_head
    Bytes for QK^T: (n_q_heads * d_head + n_kv_heads * kv_cache_len * d_head) * dtype_bytes
    
    (Hint: with GQA, multiple Q heads share the same KV head)
    """
    # YOUR CODE HERE
    pass

# Test:
# for kv_len in [128, 512, 2048, 8192, 32768]:
#     ai = attention_ai_decode(64, 8, 128, kv_len)
#     print(f"KV cache length {kv_len}: AI = {ai:.2f} FLOP/byte")

### Exercise 3: Optimal Batch Size Calculator

Write a function that, given model and GPU specs, finds the batch size that maximally utilizes both compute and memory bandwidth (i.e., the batch size that puts the operation near the ridge point).

In [ ]:
def optimal_batch_size(
    d_model: int,
    peak_tflops: float,
    peak_bw_tbs: float,
    dtype_bytes: int = 2
) -> int:
    """
    Exercise: Find the batch size that puts a linear layer at the ridge point.
    
    Ridge point = peak_tflops / peak_bw_tbs (FLOP/byte, after unit conversion)
    
    For linear layer (d x d) with batch B:
    AI = 2*B*d^2 / ((B*d + d^2 + B*d) * dtype_bytes)
    
    Set AI = ridge and solve for B.
    """
    # YOUR CODE HERE
    pass

# Test:
# bs = optimal_batch_size(4096, 312, 2.0, 2)  # A100
# print(f"Optimal batch size for A100: {bs}")
# bs = optimal_batch_size(4096, 989, 3.35, 2)  # H100
# print(f"Optimal batch size for H100: {bs}")

---

## Summary

### Key Takeaways

1. **Arithmetic Intensity** (FLOP/byte) determines whether an operation is memory-bound or compute-bound. It is the single most important metric for understanding kernel performance.

2. **The Roofline Model** provides an upper bound on performance: $P = \min(\pi, \beta \times I)$. Operations below the roof are underutilizing the hardware.

3. **LLM Prefill is Compute-Bound**: Processing many tokens at once gives high AI from large matrix multiplies.

4. **LLM Decode is Memory-Bound**: Processing one token at a time means AI ~ 1 FLOP/byte, far below the ridge point of 150-300 on modern GPUs.

5. **Batching Increases AI**: More tokens per batch amortizes weight reading, moving the operation up the memory-bound slope toward the ridge.

6. **Quantization Increases AI**: Reducing data sizes (FP16 → INT8 → INT4) reads fewer bytes for the same FLOPs, effectively doubling/quadrupling the AI.

7. **GPU Evolution**: H100 has both higher compute and bandwidth than A100, but a similar ridge point. H200 has a lower ridge point (more bandwidth-friendly), making it better for memory-bound LLM decode.

### What's Next

In **Chapter 1.4: Tokenization, Detokenization & Streaming**, we'll shift from hardware to the data pipeline, understanding how text is converted to and from tokens — a critical but often overlooked part of the serving stack.